# 01 - Dataset Preprocessing

Download Lichess standard rated games (1 month), parse PGN data, and produce:
1. **Structured CSV** with game metadata (ELO, time control, moves, openings, etc.)
2. **Blunder analysis CSV** with eval changes and clock times from games that have eval+clock annotations

Outputs are saved to Google Drive for reuse.

## Configuration
Set the month to download and processing parameters below.

In [ ]:
# ============================================================
# CONFIGURATION - Change these values as needed
# ============================================================

# Month to download (format: YYYY-MM)
# Available: 2013-01 onwards. Not all months have eval+clock data.
# Recent months tend to have more eval data.
YEAR = 2024
MONTH = 12

# Blunder threshold in centipawns (200 = same as Lichess)
BLUNDER_CUTOFF_CP = 200

# Maximum number of games to process (None = all)
MAX_GAMES = None

# Sync intermediate results to Drive every N games
SYNC_EVERY = 50_000

# Number of parallel workers for decompression
NUM_WORKERS = 4

## Setup
Install dependencies and mount Google Drive.

In [ ]:
import subprocess
import sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("python-chess")
install("zstandard")
install("pandas")
install("tqdm")

print("Dependencies installed.")

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount("/content/drive")

# Paths
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/Learning-document/Data Visualization/Project 2/data"
VM_DATA_DIR = "/content/data"

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(VM_DATA_DIR, exist_ok=True)

print(f"Drive output: {DRIVE_OUTPUT_DIR}")
print(f"VM data dir:  {VM_DATA_DIR}")

## Download
Download the zstd-compressed PGN file from Lichess.

In [ ]:
import urllib.request

month_str = f"{YEAR}-{MONTH:02d}"
filename = f"lichess_db_standard_rated_{month_str}.pgn.zst"
url = f"https://database.lichess.org/standard/{filename}"
local_path = f"/content/{filename}"

if not os.path.exists(local_path):
    print(f"Downloading {url}...")
    print("This may take 10-30 minutes depending on file size.")
    urllib.request.urlretrieve(url, local_path)
    size_gb = os.path.getsize(local_path) / (1024**3)
    print(f"Downloaded: {size_gb:.1f} GB")
else:
    print(f"File already exists: {local_path}")

## Streaming PGN Parser
Read the compressed PGN file line-by-line (no need to decompress to disk).
Extract both game metadata and move-level eval/clock data.

In [ ]:
import zstandard as zstd
import io
import chess
import chess.pgn
import re
from collections import defaultdict
from tqdm.notebook import tqdm
import pandas as pd
import time
import json

print("Libraries loaded.")

In [ ]:
def open_zst_stream(filepath):
    """Open a .zst file for streaming line-by-line reading."""
    dctx = zstd.ZstdDecompressor()
    with open(filepath, 'rb') as fh:
        stream = dctx.stream_reader(fh)
        text_stream = io.TextIOWrapper(stream, encoding='utf-8')
        yield from text_stream


def read_games_from_stream(lines):
    """Yield individual game strings from a stream of PGN lines.
    
    PGN format: header lines in [Key "Value"] format, then a blank line,
    then the movetext, then a blank line separating games.
    """
    current_game = []
    for line in lines:
        line = line.rstrip('\n')
        if line == '' and current_game:
            # Check if we have both headers and movetext
            # (empty line after headers = between headers and movetext,
            #  empty line after movetext = end of game)
            has_movetext = any(not l.startswith('[') for l in current_game)
            if has_movetext:
                yield '\n'.join(current_game)
                current_game = []
            else:
                current_game.append(line)
        else:
            if line:
                current_game.append(line)
    # Yield last game if file doesn't end with blank line
    if current_game:
        yield '\n'.join(current_game)


def parse_header(headers, key):
    """Safely extract a PGN header value."""
    try:
        return headers.get(key, None)
    except:
        return None


def time_to_seconds(clock_str):
    """Convert clock string like '0:01:30' to seconds."""
    if not clock_str:
        return None
    parts = clock_str.split(':')
    parts.reverse()
    total = 0
    mult = 1
    for p in parts:
        total += mult * float(p)
        mult *= 60
    return total


def parse_time_control(tc_str):
    """Parse time control like '300+0' or '600+5' into (base_seconds, increment)."""
    if not tc_str or '+' not in tc_str:
        return None, None
    try:
        parts = tc_str.split('+')
        return int(parts[0]), int(parts[1])
    except:
        return None, None


print("Parser functions defined.")

## Process Games
Stream through the PGN and extract:
- Game metadata for the structured CSV
- Per-move eval/clock data for blunder analysis

In [ ]:
def save_and_sync(games_metadata, blunder_records, move_records,
                  vm_dir, drive_dir, month_str):
    """Save DataFrames to VM disk, then copy to Google Drive."""
    df_g = pd.DataFrame(games_metadata)
    df_b = pd.DataFrame(blunder_records)
    df_m = pd.DataFrame(move_records)

    for suffix, df in [("games", df_g), ("blunders", df_b), ("moves", df_m)]:
        path = os.path.join(vm_dir, f"lichess_{month_str}_{suffix}.csv")
        df.to_csv(path, index=False)

    for suffix in ["games", "blunders", "moves"]:
        src = os.path.join(vm_dir, f"lichess_{month_str}_{suffix}.csv")
        dst = os.path.join(drive_dir, f"lichess_{month_str}_{suffix}.csv")
        if os.path.exists(src):
            shutil.copy2(src, dst)


def process_all_games(filepath, max_games=None, sync_every=None,
                      vm_dir=None, drive_dir=None):
    """
    Stream through the PGN file and extract:
    1. Game metadata (for structured CSV)
    2. Move-level eval+clock data (for blunder analysis)

    Every `sync_every` games, saves partial results to `vm_dir`
    and syncs to `drive_dir`.

    Returns:
        games_metadata: list of dicts with game-level info
        blunder_records: list of dicts with blunder-level info
        move_records: list of dicts with all move-level eval+clock data
    """
    month_str = f"{YEAR}-{MONTH:02d}"
    games_metadata = []
    blunder_records = []
    move_records = []

    game_count = 0
    skipped = 0
    start_time = time.time()

    dctx = zstd.ZstdDecompressor(max_window_size=2**31)

    with open(filepath, 'rb') as fh:
        stream_reader = dctx.stream_reader(fh)
        text_stream = io.TextIOWrapper(stream_reader, encoding='utf-8', errors='replace')

        while True:
            if max_games is not None and game_count >= max_games:
                break

            game = chess.pgn.read_game(text_stream)
            if game is None:
                break  # End of file

            game_count += 1
            headers = game.headers

            # --- Extract game metadata ---
            event = parse_header(headers, 'Event')
            site = parse_header(headers, 'Site')
            white_elo = parse_header(headers, 'WhiteElo')
            black_elo = parse_header(headers, 'BlackElo')
            result = parse_header(headers, 'Result')
            time_control = parse_header(headers, 'TimeControl')
            termination = parse_header(headers, 'Termination')
            opening_eco = parse_header(headers, 'ECO')
            opening_name = parse_header(headers, 'Opening')

            # Determine winner from result
            if result == '1-0':
                winner = 'white'
            elif result == '0-1':
                winner = 'black'
            elif result == '1/2-1/2':
                winner = 'draw'
            else:
                winner = None

            # Parse time control
            base_secs, increment = parse_time_control(time_control)

            # Determine victory status from termination
            victory_status = None
            if termination:
                if 'Normal' in termination:
                    victory_status = 'mate' if 'checkmate' in termination.lower() else 'resign'
                elif 'Time' in termination:
                    victory_status = 'timeout'
                elif 'Abandoned' in termination:
                    victory_status = 'abandoned'
                elif 'Rules infraction' in termination:
                    victory_status = 'rules_infraction'

            # Build movetext
            moves_list = []
            board = game.board()
            node = game

            # Track evals and clocks for blunder analysis
            has_eval = False
            has_clock = False
            prev_eval = 0.0  # centipawns from white's perspective
            move_number = 0
            total_moves = 0

            while node.variations:
                next_node = node.variation(0)
                move = next_node.move
                san = board.san(move)
                moves_list.append(san)
                total_moves += 1

                # Check for eval and clock annotations
                comment = next_node.comment
                eval_cp = None
                clock_secs = None

                # Parse eval from comment: [%eval +1.50] or [%eval #-4]
                eval_match = re.search(r'%eval\s+([^\s\]]+)', comment)
                if eval_match:
                    has_eval = True
                    eval_str = eval_match.group(1)
                    try:
                        if eval_str.startswith('#'):
                            mate_in = int(eval_str[1:])
                            eval_cp = 10000 if mate_in > 0 else -10000
                        else:
                            eval_cp = int(float(eval_str) * 100)
                    except:
                        eval_cp = None

                # Parse clock from comment: [%clk 0:01:30]
                clk_match = re.search(r'%clk\s+([^\s\]]+)', comment)
                if clk_match:
                    has_clock = True
                    clock_secs = time_to_seconds(clk_match.group(1))

                # Determine whose turn it is
                is_white = board.turn == chess.WHITE
                move_number += 1

                # Record move-level data if we have eval+clock
                if eval_cp is not None and clock_secs is not None:
                    if is_white:
                        eval_change = eval_cp - prev_eval
                    else:
                        eval_change = prev_eval - eval_cp

                    move_records.append({
                        'game_idx': game_count - 1,
                        'move_number': move_number,
                        'is_white': is_white,
                        'move_san': san,
                        'eval_cp': eval_cp,
                        'eval_change_for_player': eval_change,
                        'clock_secs_remaining': clock_secs,
                        'base_time_secs': base_secs,
                        'increment': increment,
                    })

                    # Check for blunder
                    if eval_change <= -BLUNDER_CUTOFF_CP:
                        was_winning = (is_white and prev_eval > 200) or (not is_white and prev_eval < -200)
                        still_winning = (is_white and eval_cp > 200) or (not is_white and eval_cp < -200)

                        if not (was_winning and still_winning):
                            blunder_records.append({
                                'game_idx': game_count - 1,
                                'move_number': move_number,
                                'is_white': is_white,
                                'move_san': san,
                                'eval_change_cp': eval_change,
                                'clock_secs_remaining': clock_secs,
                                'base_time_secs': base_secs,
                                'white_elo': int(white_elo) if white_elo else None,
                                'black_elo': int(black_elo) if black_elo else None,
                                'player_elo': int(white_elo) if is_white else int(black_elo) if black_elo else None,
                            })

                    prev_eval = eval_cp

                board.push(move)
                node = next_node

            # Determine if game was rated
            event_str = event or ''
            is_rated = 'Rated' in event_str

            # Determine game category
            game_type = None
            if base_secs is not None:
                total_est = base_secs + increment * 80 if increment else base_secs
                if total_est < 180:
                    game_type = 'bullet'
                elif total_est < 480:
                    game_type = 'blitz'
                elif total_est < 1500:
                    game_type = 'rapid'
                else:
                    game_type = 'classical'

            games_metadata.append({
                'game_id': site.split('/')[-1] if site else None,
                'rated': is_rated,
                'turns': total_moves,
                'victory_status': victory_status,
                'winner': winner,
                'white_id': parse_header(headers, 'White'),
                'white_rating': int(white_elo) if white_elo and white_elo != '?' else None,
                'black_id': parse_header(headers, 'Black'),
                'black_rating': int(black_elo) if black_elo and black_elo != '?' else None,
                'moves': ' '.join(moves_list),
                'increment_code': time_control,
                'base_time_secs': base_secs,
                'increment': increment,
                'game_type': game_type,
                'opening_eco': opening_eco,
                'opening_name': opening_name,
                'opening_ply': sum(1 for _ in game.mainline_moves()) // 2,
                'has_eval': has_eval,
                'has_clock': has_clock,
            })

            # Periodic sync: save to VM then copy to Drive
            if sync_every and game_count % sync_every == 0:
                elapsed = time.time() - start_time
                rate = game_count / elapsed
                save_and_sync(games_metadata, blunder_records, move_records,
                              vm_dir, drive_dir, month_str)
                print(f"  Processed {game_count:,} games ({rate:.0f} games/sec) - synced to VM & Drive")

    # Final save
    if vm_dir:
        save_and_sync(games_metadata, blunder_records, move_records,
                      vm_dir, drive_dir, month_str)

    elapsed = time.time() - start_time
    print(f"\nDone: {game_count:,} games processed in {elapsed:.1f}s")

    return games_metadata, blunder_records, move_records

In [ ]:
print(f"Processing {YEAR}-{MONTH:02d} Lichess standard rated games...")
if MAX_GAMES:
    print(f"Limit: {MAX_GAMES:,} games")
else:
    print("Processing ALL games (this may take 1-3 hours)")

games_metadata, blunder_records, move_records = process_all_games(
    local_path,
    max_games=MAX_GAMES,
    sync_every=SYNC_EVERY,
    vm_dir=VM_DATA_DIR,
    drive_dir=DRIVE_OUTPUT_DIR,
)

print(f"\nResults:")
print(f"  Games metadata: {len(games_metadata):,}")
print(f"  Blunder records: {len(blunder_records):,}")
print(f"  Move records (with eval+clock): {len(move_records):,}")

## Save to Google Drive

In [ ]:
# Create DataFrames
df_games = pd.DataFrame(games_metadata)
df_blunders = pd.DataFrame(blunder_records)
df_moves = pd.DataFrame(move_records)

# Preview
print("=== Games Metadata ===")
print(f"Shape: {df_games.shape}")
print(df_games.head())

print("\n=== Blunder Records ===")
print(f"Shape: {df_blunders.shape}")
if not df_blunders.empty:
    print(df_blunders.head())
else:
    print("No blunder records (games may lack eval+clock data)")

print("\n=== Move Records (eval+clock) ===")
print(f"Shape: {df_moves.shape}")
if not df_moves.empty:
    print(df_moves.head())
else:
    print("No move records (games may lack eval+clock data)")

In [ ]:
# Final save to VM and sync to Drive
month_str = f"{YEAR}-{MONTH:02d}"

save_and_sync(games_metadata, blunder_records, move_records,
              VM_DATA_DIR, DRIVE_OUTPUT_DIR, month_str)

for suffix in ["games", "blunders", "moves"]:
    path = os.path.join(VM_DATA_DIR, f"lichess_{month_str}_{suffix}.csv")
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024**2
        print(f"Saved: {path} ({size_mb:.1f} MB)")
    else:
        print(f"No data: {path}")

## Summary Statistics

In [ ]:
print(f"=== Summary for {month_str} ===")
print(f"Total games: {len(df_games):,}")
print(f"\nGame type breakdown:")
if 'game_type' in df_games.columns:
    print(df_games['game_type'].value_counts().to_string())

print(f"\nWinner breakdown:")
print(df_games['winner'].value_counts().to_string())

print(f"\nELO stats (white):")
if df_games['white_rating'].notna().any():
    print(df_games['white_rating'].describe().to_string())

print(f"\nGames with eval data: {df_games['has_eval'].sum():,} ({df_games['has_eval'].mean()*100:.1f}%)")
print(f"Games with clock data: {df_games['has_clock'].sum():,} ({df_games['has_clock'].mean()*100:.1f}%)")

if not df_blunders.empty:
    print(f"\nTotal blunders detected: {len(df_blunders):,}")
    print(f"Blunder rate: {len(df_blunders) / max(len(df_moves), 1) * 100:.2f}% of moves")